In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [6]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [23]:
PrevLoans = pd.read_parquet('a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\prev_loan_data_processed.parquet')

In [24]:
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

In [8]:
CCBal = pd.read_csv('data/Internal/credit_card_balance.csv')

In [9]:
CCBal = CCBal.astype({'SK_ID_CURR': 'string', 'SK_ID_PREV': 'string'})
CCBal.dtypes

SK_ID_PREV                    string[python]
SK_ID_CURR                    string[python]
MONTHS_BALANCE                         int64
AMT_BALANCE                          float64
AMT_CREDIT_LIMIT_ACTUAL                int64
AMT_DRAWINGS_ATM_CURRENT             float64
AMT_DRAWINGS_CURRENT                 float64
AMT_DRAWINGS_OTHER_CURRENT           float64
AMT_DRAWINGS_POS_CURRENT             float64
AMT_INST_MIN_REGULARITY              float64
AMT_PAYMENT_CURRENT                  float64
AMT_PAYMENT_TOTAL_CURRENT            float64
AMT_RECEIVABLE_PRINCIPAL             float64
AMT_RECIVABLE                        float64
AMT_TOTAL_RECEIVABLE                 float64
CNT_DRAWINGS_ATM_CURRENT             float64
CNT_DRAWINGS_CURRENT                   int64
CNT_DRAWINGS_OTHER_CURRENT           float64
CNT_DRAWINGS_POS_CURRENT             float64
CNT_INSTALMENT_MATURE_CUM            float64
NAME_CONTRACT_STATUS                  object
SK_DPD                                 int64
SK_DPD_DEF

In [10]:
CCBal.sort_values(by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE'], ascending=[True, True, True], inplace=True)
CCBal = CCBal.reset_index(drop=True)

In [ ]:
#temp = CCBal.groupby('SK_ID_PREV')[['MONTHS_BALANCE']].agg(['min', 'max']).reset_index()

In [11]:
CCBal = base.merge(CCBal, on='SK_ID_CURR', how='inner')

In [8]:
CCBal = CCBal[CCBal['TARGET'].notna()]

In [ ]:
#temp = CCBal[['SK_ID_PREV','SK_ID_CURR']].drop_duplicates().groupby('SK_ID_CURR').size().reset_index(name='count')
#temp = temp[temp['count'] > 1]

In [7]:
def create_yoy_features(df, pct_cols, abs_cols):
    """
    Engineers Year-over-Year (YoY) change features for a credit dataset.

    This function calculates the YoY percentage change for financial amounts and
    the YoY absolute difference for metrics like 'days past due'.

    Args:
        df (pd.DataFrame): The input DataFrame containing credit history. It must contain 'SK_ID_CURR' for client ID, 'MONTHS_BALANCE' for the timeline, and the metrics to be transformed.
        
        amount_cols (list): List of columns for which we want to calculate YoY percentage change. These are typically monetary values where relative change is important.

        other_cols (list): List of columns for which we want to calculate YoY absolute difference. These are often counts or statuses where the raw change is more informative.

    Returns:
        pd.DataFrame: The DataFrame with new YoY feature columns added.
    """
    # --- Pre-computation & Sorting ---
    # For this calculation to work correctly, the data must be sorted by the
    # individual credit account (SK_ID_CURR) and the timeline (MONTHS_BALANCE).

    # --- Feature Engineering ---
    # We group by each client to ensure the YoY calculation is done within
    # the same credit account.
    df = df.copy()
    grouped = df.groupby(['SK_ID_CURR'])
    df[pct_cols] = df[pct_cols].clip(lower=0) 

    # --- Calculate YoY Percentage Change for Amounts ---
    for col in pct_cols:
        # Get the value from 12 months ago. The shift(-12) function moves the
        # data down up 12 rows within each group.
        new_val = grouped[col].shift(3)

        # Calculate the YoY percentage change.
        # (current - past) / past
        # We use np.where to avoid division by zero if the past value was 0.
        # If the denominator is 0, we'll record a large change (1) if the current
        # value is positive, and 0 otherwise.
        df[f'{col}_YOY_PCT_CHANGE'] = (df[col] - new_val) / new_val.replace(0, np.nan) #np.where(new_val == 0, 1, new_val)
        # Replace infinite values (from division by a very small number) with NaN
        # df[f'{col}_YOY_PCT_CHANGE'].clip(lower=-1, upper=1, inplace=True)
        df[f'{col}_YOY_PCT_CHANGE'] = df[f'{col}_YOY_PCT_CHANGE'].replace([np.inf, -np.inf], np.nan)


    # --- Calculate YoY Absolute Difference for Other Metrics ---
    for col in abs_cols:
        # Get the value from 12 months ago.
        prev_year_val = grouped[col].shift(3)

        # Calculate the absolute difference: current_value - past_value
        df[f'{col}_YOY_ABS_CHANGE'] = df[col] - prev_year_val

    # --- Final Touches ---
    # The shift operation creates NaNs for the first 12 months of data for each
    # account. For modeling, it's often best to fill these with 0, assuming
    # no change when historical data is not available.
    yoy_cols = [col for col in df.columns if '_YOY_' in col]
    # df[yoy_cols] = df[yoy_cols].fillna(0)

    print("YoY features created successfully. New columns:")
    print(yoy_cols)

    return df[['SK_ID_CURR','MONTHS_BALANCE']+yoy_cols]


Impute data

In [13]:
CCBal.loc[(CCBal['AMT_DRAWINGS_CURRENT'] == 0) & (CCBal['AMT_DRAWINGS_ATM_CURRENT'].isna()), 'AMT_DRAWINGS_ATM_CURRENT'] = 0
CCBal.loc[(CCBal['AMT_DRAWINGS_CURRENT'] == 0) & (CCBal['AMT_DRAWINGS_OTHER_CURRENT'].isna()), 'AMT_DRAWINGS_OTHER_CURRENT'] = 0
CCBal.loc[(CCBal['AMT_DRAWINGS_CURRENT'] == 0) & (CCBal['AMT_DRAWINGS_POS_CURRENT'].isna()), 'AMT_DRAWINGS_POS_CURRENT'] = 0

In [17]:
CCBal[CCBal['AMT_DRAWINGS_CURRENT'] > 0][CCBal['AMT_DRAWINGS_ATM_CURRENT'] > 0] # AMT_DRAWINGS_OTHER_CURRENT AMT_DRAWINGS_POS_CURRENT

C:\Users\domjo\AppData\Local\Temp\ipykernel_43588\4122397814.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  CCBal[CCBal['AMT_DRAWINGS_CURRENT'] > 0][CCBal['AMT_DRAWINGS_ATM_CURRENT'] > 0] # AMT_DRAWINGS_OTHER_CURRENT AMT_DRAWINGS_POS_CURRENT


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF,TARGET
6,1843384,100011,-75,189000.000,180000,180000.0,180000.000,0.0,0.000,9000.00,...,189000.000,4.0,4,0.0,0.0,NaN,Active,0,0,0.0
354,1557583,100043,-25,435861.585,427500,67500.0,67500.000,0.0,0.000,18361.17,...,433170.585,2.0,2,0.0,0.0,7.0,Active,0,0,0.0
444,1060558,100048,-2,27302.400,225000,27000.0,27000.000,0.0,0.000,0.00,...,25956.900,1.0,1,0.0,0.0,17.0,Active,0,0,0.0
452,1209367,100049,-8,10397.115,90000,9000.0,9000.000,0.0,0.000,0.00,...,9051.615,1.0,1,0.0,0.0,0.0,Active,0,0,1.0
453,1209367,100049,-7,82449.135,90000,72000.0,73990.935,0.0,1990.935,0.00,...,78412.635,3.0,6,0.0,3.0,0.0,Active,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3840185,1595171,456247,-93,186025.230,180000,96750.0,96750.000,0.0,0.000,4500.00,...,186025.230,4.0,4,0.0,0.0,6.0,Active,0,0,0.0
3840190,1595171,456247,-88,185570.505,180000,9000.0,9000.000,0.0,0.000,9000.00,...,185570.505,2.0,2,0.0,0.0,11.0,Active,0,0,0.0
3840199,1595171,456247,-79,4143.420,180000,6750.0,6750.000,0.0,0.000,0.00,...,4143.420,2.0,2,0.0,0.0,17.0,Active,0,0,0.0
3840202,1595171,456247,-76,96776.640,180000,90000.0,90000.000,0.0,0.000,0.00,...,96776.640,2.0,2,0.0,0.0,19.0,Active,0,0,0.0


In [15]:
CCBal.loc[(CCBal['CNT_DRAWINGS_CURRENT'] == 0) & (CCBal['CNT_DRAWINGS_ATM_CURRENT'].isna()), 'CNT_DRAWINGS_ATM_CURRENT'] = 0
CCBal.loc[(CCBal['CNT_DRAWINGS_CURRENT'] == 0) & (CCBal['CNT_DRAWINGS_POS_CURRENT'].isna()), 'CNT_DRAWINGS_POS_CURRENT'] = 0
CCBal.loc[(CCBal['CNT_DRAWINGS_CURRENT'] == 0) & (CCBal['CNT_DRAWINGS_OTHER_CURRENT'].isna()), 'CNT_DRAWINGS_OTHER_CURRENT'] = 0

In [16]:
CCBal['AMT_PAYMENT_CURRENT'] = CCBal.groupby('SK_ID_PREV')['AMT_PAYMENT_CURRENT'].transform(
    lambda x: x.ffill(limit=6).bfill(limit=6)
) #.reset_index()

CCBal['AMT_INST_MIN_REGULARITY'] = CCBal.groupby('SK_ID_PREV')['AMT_INST_MIN_REGULARITY'].transform(
    lambda x: x.ffill(limit=6).bfill(limit=6)
) 

Number of Previous Accounts

In [17]:
CCBal['UTILIZATION_RATIO_PREVLOAN'] = np.where(CCBal['AMT_CREDIT_LIMIT_ACTUAL'] >= 0, 
                                     ( CCBal['AMT_BALANCE'] / (1 + CCBal['AMT_CREDIT_LIMIT_ACTUAL']) ) * 100, 
                                     np.nan)
CCBal['PAYMENT_RATIO_PREVLOAN'] = np.where((CCBal['AMT_PAYMENT_TOTAL_CURRENT'] + CCBal['AMT_TOTAL_RECEIVABLE']) >= 0 , 
                                 ( CCBal['AMT_PAYMENT_TOTAL_CURRENT'] / (1 + CCBal['AMT_PAYMENT_TOTAL_CURRENT'] + CCBal['AMT_TOTAL_RECEIVABLE']) ) * 100 ,
                                 np.nan)

CCBal['DRAWING_TO_PYMNT_RATIO_PREVLOAN'] = np.where(CCBal['AMT_PAYMENT_TOTAL_CURRENT'] >= 0 , 
                                          ( CCBal['AMT_DRAWINGS_CURRENT'] / ( CCBal['AMT_PAYMENT_TOTAL_CURRENT'] + 1) ) * 100 ,
                                          np.nan)

CCBal['INT_COVERAGE_RATIO_PREVLOAN']    = np.where(CCBal['AMT_PAYMENT_TOTAL_CURRENT'] >= 0 , 
                                         ((CCBal['AMT_TOTAL_RECEIVABLE']-CCBal['AMT_RECEIVABLE_PRINCIPAL']) / ( CCBal['AMT_PAYMENT_TOTAL_CURRENT'] + 1) ) * 100 ,
                                         np.nan)   
CCBal['POS_DRAWING_AMT_RATIO_PREVLOAN'] = np.where(CCBal['AMT_DRAWINGS_CURRENT'] >= 0 , 
                                         (CCBal['AMT_DRAWINGS_POS_CURRENT'] / (1 + CCBal['AMT_DRAWINGS_CURRENT']) ) * 100, 
                                         np.nan)

CCBal['ATM_DRAWING_AMT_RATIO_PREVLOAN'] = np.where(CCBal['AMT_DRAWINGS_CURRENT'] >= 0 , 
                                         (CCBal['AMT_DRAWINGS_ATM_CURRENT'] / (1 + CCBal['AMT_DRAWINGS_CURRENT']) ) * 100  , 
                                         np.nan)

CCBal['POS_DRAWING_CNT_RATIO_PREVLOAN'] = np.where(CCBal['CNT_DRAWINGS_CURRENT'] >= 0 , 
                                         (CCBal['CNT_DRAWINGS_POS_CURRENT'] / (1 + CCBal['CNT_DRAWINGS_CURRENT']) ) * 100, 
                                         np.nan)
  
CCBal['ATM_DRAWING_CNT_RATIO_PREVLOAN'] = np.where(CCBal['CNT_DRAWINGS_CURRENT'] >= 0 , 
                                         ( CCBal['CNT_DRAWINGS_ATM_CURRENT'] / (1 + CCBal['CNT_DRAWINGS_CURRENT']) ) * 100, 
                                         np.nan)  

In [ ]:
CCBal_Prev_ID = (
    CCBal.groupby(['SK_ID_CURR', 'SK_ID_PREV'])
      .agg(
          COMPLETED_PREV_LOAN=('NAME_CONTRACT_STATUS', lambda x: (x == 'Completed').any().sum()),
          AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN=('AMT_CREDIT_LIMIT_ACTUAL', 'first'),
          AMT_CREDIT_LIMIT_ACTUAL_FINAL_PREVLOAN=('AMT_CREDIT_LIMIT_ACTUAL', 'last'),
          AMT_BALANCE_AVG_PREVLOAN=('AMT_BALANCE', 'mean'),
          AMT_BALANCE_MAX_PREVLOAN=('AMT_BALANCE', 'max'),
          AVG_UTILIZATION_RATIO_PREVLOAN=('UTILIZATION_RATIO_PREVLOAN', 'mean'),
          AVG_PAYMENT_RATIO_PREVLOAN=('PAYMENT_RATIO_PREVLOAN', 'mean'),
          AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN=('DRAWING_TO_PYMNT_RATIO_PREVLOAN', 'mean'),
          AVG_INT_COVERAGE_RATIO_PREVLOAN=('INT_COVERAGE_RATIO_PREVLOAN', 'mean'),
          AVG_POS_DRAWING_AMT_RATIO_PREVLOAN=('POS_DRAWING_AMT_RATIO_PREVLOAN', 'mean'),
          AVG_ATM_DRAWING_AMT_RATIO_PREVLOAN=('ATM_DRAWING_AMT_RATIO_PREVLOAN', 'mean'),
          AVG_POS_DRAWING_CNT_RATIO_PREVLOAN=('POS_DRAWING_CNT_RATIO_PREVLOAN', 'mean'),
          AVG_ATM_DRAWING_CNT_RATIO_PREVLOAN=('ATM_DRAWING_CNT_RATIO_PREVLOAN', 'mean'),
          Duration = ('MONTHS_BALANCE', lambda x: x.max() - x.min())
      )
      .reset_index()
)

In [27]:
CCBal_Prev_ID = CCBal_Prev_ID.merge(PrevLoans, on=['SK_ID_CURR','SK_ID_PREV'], how='left')

In [ ]:
CCBal_Prev_ID_Counts = CCBal_Prev_ID.groupby('SK_ID_CURR').agg(
    CC_TOTAL_PREV_LOAN = ('SK_ID_PREV', 'nunique'),   
    CC_COMPLETED_PREV_LOAN = ('COMPLETED_PREV_LOAN', 'sum'),

    CC_MAX_AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN = ('AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN', 'max'),
    CC_MAX_AMT_CREDIT_LIMIT_ACTUAL_FINAL_PREVLOAN = ('AMT_CREDIT_LIMIT_ACTUAL_FINAL_PREVLOAN', 'max'),
    CC_MAX_AMT_BALANCE_AVG_PREVLOAN = ('AMT_BALANCE_AVG_PREVLOAN', 'max'),
    CC_AVG_AMT_BALANCE_AVG_PREVLOAN = ('AMT_BALANCE_AVG_PREVLOAN', 'mean'),

    CC_MAX_UTILIZATION_RATIO_PREVLOAN = ('AVG_UTILIZATION_RATIO_PREVLOAN', 'max'),
    CC_MAX_PAYMENT_RATIO_PREVLOAN = ('AVG_PAYMENT_RATIO_PREVLOAN', 'max'),
    CC_MAX_DRAWING_TO_PYMNT_RATIO_PREVLOAN = ('AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN', 'max'),
    CC_MAX_INT_COVERAGE_RATIO_PREVLOAN = ('AVG_INT_COVERAGE_RATIO_PREVLOAN', 'max'),
    CC_MAX_POS_DRAWING_AMT_RATIO_PREVLOAN = ('AVG_POS_DRAWING_AMT_RATIO_PREVLOAN', 'max'),
    CC_MAX_ATM_DRAWING_AMT_RATIO_PREVLOAN = ('AVG_ATM_DRAWING_AMT_RATIO_PREVLOAN', 'max'),
    CC_MAX_POS_DRAWING_CNT_RATIO_PREVLOAN = ('AVG_POS_DRAWING_CNT_RATIO_PREVLOAN', 'max'),
    CC_MAX_ATM_DRAWING_CNT_RATIO_PREVLOAN = ('AVG_ATM_DRAWING_CNT_RATIO_PREVLOAN', 'max'),

    CC_AVG_UTILIZATION_RATIO_PREVLOAN = ('AVG_UTILIZATION_RATIO_PREVLOAN', 'mean'),
    CC_AVG_PAYMENT_RATIO_PREVLOAN = ('AVG_PAYMENT_RATIO_PREVLOAN', 'mean'),
    CC_AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN = ('AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN', 'mean'),
    CC_AVG_INT_COVERAGE_RATIO_PREVLOAN = ('AVG_INT_COVERAGE_RATIO_PREVLOAN', 'mean'),
    CC_AVG_POS_DRAWING_AMT_RATIO_PREVLOAN = ('AVG_POS_DRAWING_AMT_RATIO_PREVLOAN', 'mean'),
    CC_AVG_ATM_DRAWING_AMT_RATIO_PREVLOAN = ('AVG_ATM_DRAWING_AMT_RATIO_PREVLOAN', 'mean'),
    CC_AVG_POS_DRAWING_CNT_RATIO_PREVLOAN = ('AVG_POS_DRAWING_CNT_RATIO_PREVLOAN', 'mean'),
    CC_AVG_ATM_DRAWING_CNT_RATIO_PREVLOAN = ('AVG_ATM_DRAWING_CNT_RATIO_PREVLOAN', 'mean'),
    
    CC_MAX_DURATION_PREVLOAN = ('Duration', 'max'),
    CC_AVG_DURATION_PREVLOAN = ('Duration', 'mean'),

    CC_CNT_NFLAG_INSURED_ON_APPROVAL_PREVLOAN = ('NFLAG_INSURED_ON_APPROVAL', 'sum'),
    CC_AVG_DAYS_DECISION_PREVLOAN = ('DAYS_DECISION', 'mean')

)

In [ ]:
del CCBal_Prev_ID, PrevLoans

**Monthly Data**

In [ ]:
sum_cols = [
    'AMT_BALANCE',
    'AMT_CREDIT_LIMIT_ACTUAL', 
    'AMT_DRAWINGS_ATM_CURRENT',
    'AMT_DRAWINGS_CURRENT', 
    'AMT_DRAWINGS_OTHER_CURRENT',
    'AMT_DRAWINGS_POS_CURRENT', 
    'AMT_INST_MIN_REGULARITY',
    'AMT_PAYMENT_CURRENT', 
    'AMT_PAYMENT_TOTAL_CURRENT',
    'AMT_RECEIVABLE_PRINCIPAL', 
    'AMT_RECIVABLE', 
    'AMT_TOTAL_RECEIVABLE',
    'CNT_DRAWINGS_ATM_CURRENT', 
    'CNT_DRAWINGS_CURRENT',
    'CNT_DRAWINGS_OTHER_CURRENT', 
    'CNT_DRAWINGS_POS_CURRENT',
    'CNT_INSTALMENT_MATURE_CUM'
]

In [75]:
Sum_CCBal_Mon = CCBal.groupby(['SK_ID_CURR','MONTHS_BALANCE'])[sum_cols].sum().reset_index()

Create Ratios

In [ ]:
Sum_CCBal_Mon['UTILIZATION_RATIO'] = np.where(Sum_CCBal_Mon['AMT_CREDIT_LIMIT_ACTUAL'] >= 0, 
                                     ( Sum_CCBal_Mon['AMT_BALANCE'] / (1 + Sum_CCBal_Mon['AMT_CREDIT_LIMIT_ACTUAL']) ) * 100, 
                                     np.nan)
Sum_CCBal_Mon['PAYMENT_RATIO'] = np.where((Sum_CCBal_Mon['AMT_PAYMENT_TOTAL_CURRENT'] + Sum_CCBal_Mon['AMT_TOTAL_RECEIVABLE']) >= 0 , 
                                 ( Sum_CCBal_Mon['AMT_PAYMENT_TOTAL_CURRENT'] / (1 + Sum_CCBal_Mon['AMT_PAYMENT_TOTAL_CURRENT'] + Sum_CCBal_Mon['AMT_TOTAL_RECEIVABLE']) ) * 100 ,
                                 np.nan)

Sum_CCBal_Mon['DRAWING_TO_PYMNT_RATIO'] = np.where(Sum_CCBal_Mon['AMT_PAYMENT_TOTAL_CURRENT'] >= 0 , 
                                          ( Sum_CCBal_Mon['AMT_DRAWINGS_CURRENT'] / ( Sum_CCBal_Mon['AMT_PAYMENT_TOTAL_CURRENT'] + 1) ) * 100 ,
                                          np.nan)

Sum_CCBal_Mon['INT_COVERAGE_RATIO']    = np.where(Sum_CCBal_Mon['AMT_PAYMENT_TOTAL_CURRENT'] >= 0 , 
                                         ((Sum_CCBal_Mon['AMT_TOTAL_RECEIVABLE']-Sum_CCBal_Mon['AMT_RECEIVABLE_PRINCIPAL']) / ( Sum_CCBal_Mon['AMT_PAYMENT_TOTAL_CURRENT'] + 1) ) * 100 ,
                                         np.nan)   
Sum_CCBal_Mon['POS_DRAWING_AMT_RATIO'] = np.where(Sum_CCBal_Mon['AMT_DRAWINGS_CURRENT'] >= 0 , 
                                         (Sum_CCBal_Mon['AMT_DRAWINGS_POS_CURRENT'] / (1 + Sum_CCBal_Mon['AMT_DRAWINGS_CURRENT']) ) * 100, 
                                         np.nan)

Sum_CCBal_Mon['ATM_DRAWING_AMT_RATIO'] = np.where(Sum_CCBal_Mon['AMT_DRAWINGS_CURRENT'] >= 0 , 
                                         (Sum_CCBal_Mon['AMT_DRAWINGS_ATM_CURRENT'] / (1 + Sum_CCBal_Mon['AMT_DRAWINGS_CURRENT']) ) * 100  , 
                                         np.nan)

Sum_CCBal_Mon['POS_DRAWING_CNT_RATIO'] = np.where(Sum_CCBal_Mon['CNT_DRAWINGS_CURRENT'] >= 0 , 
                                         (Sum_CCBal_Mon['CNT_DRAWINGS_POS_CURRENT'] / (1 + Sum_CCBal_Mon['CNT_DRAWINGS_CURRENT']) ) * 100, 
                                         np.nan)
  
Sum_CCBal_Mon['ATM_DRAWING_CNT_RATIO'] = np.where(Sum_CCBal_Mon['CNT_DRAWINGS_CURRENT'] >= 0 , 
                                         ( Sum_CCBal_Mon['CNT_DRAWINGS_ATM_CURRENT'] / (1 + Sum_CCBal_Mon['CNT_DRAWINGS_CURRENT']) ) * 100, 
                                         np.nan)  
Sum_CCBal_Mon['ISNULL_DRAWINGS_CURRENT_FLAG'] = (Sum_CCBal_Mon['AMT_DRAWINGS_CURRENT'] == 0)* 1

In [77]:
Sum_CCBal_Mon.columns

Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'AMT_BALANCE',
       'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT',
       'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT',
       'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY',
       'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT',
       'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE',
       'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT',
       'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT',
       'CNT_INSTALMENT_MATURE_CUM', 'UTILIZATION_RATIO', 'PAYMENT_RATIO',
       'DRAWING_TO_PYMNT_RATIO', 'INT_COVERAGE_RATIO', 'POS_DRAWING_AMT_RATIO',
       'ATM_DRAWING_AMT_RATIO', 'POS_DRAWING_CNT_RATIO',
       'ATM_DRAWING_CNT_RATIO', 'ISNULL_DRAWINGS_CURRENT_FLAG'],
      dtype='object')

YOY data

In [66]:
other_cols = [
        'CNT_DRAWINGS_ATM_CURRENT',
        'CNT_DRAWINGS_CURRENT',
        'CNT_DRAWINGS_OTHER_CURRENT', 
        'CNT_DRAWINGS_POS_CURRENT',
        'CNT_INSTALMENT_MATURE_CUM'
]
amount_cols = [
        'AMT_BALANCE',
        'AMT_CREDIT_LIMIT_ACTUAL',
        'AMT_DRAWINGS_CURRENT',
        'AMT_DRAWINGS_ATM_CURRENT',
        'AMT_DRAWINGS_POS_CURRENT',
        'AMT_DRAWINGS_OTHER_CURRENT',
        'AMT_PAYMENT_CURRENT',
        'AMT_PAYMENT_TOTAL_CURRENT',
        'AMT_RECEIVABLE_PRINCIPAL', 
        'AMT_RECIVABLE',        
        'AMT_TOTAL_RECEIVABLE',
        'UTILIZATION_RATIO', 
        'PAYMENT_RATIO',
        'DRAWING_TO_PYMNT_RATIO', 
        'INT_COVERAGE_RATIO', 
        'POS_DRAWING_AMT_RATIO',
        'ATM_DRAWING_AMT_RATIO', 
        'POS_DRAWING_CNT_RATIO',
        'ATM_DRAWING_CNT_RATIO'
]

In [78]:
YoY_CCBal_Mon = create_yoy_features(Sum_CCBal_Mon, amount_cols, other_cols)

YoY features created successfully. New columns:
['AMT_BALANCE_YOY_PCT_CHANGE', 'AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE', 'AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE', 'AMT_DRAWINGS_ATM_CURRENT_YOY_PCT_CHANGE', 'AMT_DRAWINGS_POS_CURRENT_YOY_PCT_CHANGE', 'AMT_DRAWINGS_OTHER_CURRENT_YOY_PCT_CHANGE', 'AMT_PAYMENT_CURRENT_YOY_PCT_CHANGE', 'AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE', 'AMT_RECEIVABLE_PRINCIPAL_YOY_PCT_CHANGE', 'AMT_RECIVABLE_YOY_PCT_CHANGE', 'AMT_TOTAL_RECEIVABLE_YOY_PCT_CHANGE', 'UTILIZATION_RATIO_YOY_PCT_CHANGE', 'PAYMENT_RATIO_YOY_PCT_CHANGE', 'DRAWING_TO_PYMNT_RATIO_YOY_PCT_CHANGE', 'INT_COVERAGE_RATIO_YOY_PCT_CHANGE', 'POS_DRAWING_AMT_RATIO_YOY_PCT_CHANGE', 'ATM_DRAWING_AMT_RATIO_YOY_PCT_CHANGE', 'POS_DRAWING_CNT_RATIO_YOY_PCT_CHANGE', 'ATM_DRAWING_CNT_RATIO_YOY_PCT_CHANGE', 'CNT_DRAWINGS_ATM_CURRENT_YOY_ABS_CHANGE', 'CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE', 'CNT_DRAWINGS_OTHER_CURRENT_YOY_ABS_CHANGE', 'CNT_DRAWINGS_POS_CURRENT_YOY_ABS_CHANGE', 'CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CH

In [80]:
Sum_CCBal_Mon.columns

Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'AMT_BALANCE',
       'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT',
       'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT',
       'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY',
       'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT',
       'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE',
       'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT',
       'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT',
       'CNT_INSTALMENT_MATURE_CUM', 'UTILIZATION_RATIO', 'PAYMENT_RATIO',
       'DRAWING_TO_PYMNT_RATIO', 'INT_COVERAGE_RATIO', 'POS_DRAWING_AMT_RATIO',
       'ATM_DRAWING_AMT_RATIO', 'POS_DRAWING_CNT_RATIO',
       'ATM_DRAWING_CNT_RATIO', 'ISNULL_DRAWINGS_CURRENT_FLAG'],
      dtype='object')

**Maximum Values**

In [36]:
max_cols = [
    'SK_DPD',
    'SK_DPD_DEF'
]

In [81]:
Max_CCBal_Mon = CCBal.groupby(['SK_ID_CURR','MONTHS_BALANCE'])[max_cols].max().reset_index()

In [82]:
Max_CCBal_Mon['DPD_GE_1'] = ( Max_CCBal_Mon['SK_DPD'] >= 1 ) * 1
Max_CCBal_Mon['DPD_GE_7'] = ( Max_CCBal_Mon['SK_DPD'] >= 7 ) * 1
Max_CCBal_Mon['DPD_GE_30'] = ( Max_CCBal_Mon['SK_DPD'] >= 30 ) * 1
#Max_CCBal_Mon['DPD_UTP'] = ( (Max_CCBal_Mon['SK_DPD'] >= 1) & (Max_CCBal_Mon['AMT_BALANCE'] >= 100) ) * 1

In [16]:
# drop columns with very high NaN values
#CCBal.drop(columns=['POS_DRAWING_AMT_RATIO','ATM_DRAWING_AMT_RATIO', 'POS_DRAWING_CNT_RATIO', 'ATM_DRAWING_CNT_RATIO'], inplace=True)

In [83]:
Sum_CCBal_Mon.columns

Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'AMT_BALANCE',
       'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT',
       'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT',
       'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY',
       'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT',
       'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE',
       'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT',
       'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT',
       'CNT_INSTALMENT_MATURE_CUM', 'UTILIZATION_RATIO', 'PAYMENT_RATIO',
       'DRAWING_TO_PYMNT_RATIO', 'INT_COVERAGE_RATIO', 'POS_DRAWING_AMT_RATIO',
       'ATM_DRAWING_AMT_RATIO', 'POS_DRAWING_CNT_RATIO',
       'ATM_DRAWING_CNT_RATIO', 'ISNULL_DRAWINGS_CURRENT_FLAG'],
      dtype='object')

In [84]:
CCBal_Mon = Sum_CCBal_Mon.merge(YoY_CCBal_Mon, on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left'
                                ).merge(Max_CCBal_Mon, on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left')

In [100]:
agg_cols = [
'AMT_BALANCE',
'AMT_CREDIT_LIMIT_ACTUAL', 
'AMT_DRAWINGS_ATM_CURRENT',
'AMT_DRAWINGS_CURRENT', 
'AMT_DRAWINGS_OTHER_CURRENT',
'AMT_DRAWINGS_POS_CURRENT', 
'AMT_INST_MIN_REGULARITY',
'AMT_PAYMENT_CURRENT', 
'AMT_PAYMENT_TOTAL_CURRENT',
'AMT_RECEIVABLE_PRINCIPAL', 
'AMT_RECIVABLE',
'AMT_TOTAL_RECEIVABLE', 
'CNT_DRAWINGS_ATM_CURRENT',
'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT',
'CNT_DRAWINGS_POS_CURRENT', 'CNT_INSTALMENT_MATURE_CUM',
'UTILIZATION_RATIO', 'PAYMENT_RATIO', 'DRAWING_TO_PYMNT_RATIO',
'INT_COVERAGE_RATIO', 'POS_DRAWING_AMT_RATIO',
'ATM_DRAWING_AMT_RATIO', 'POS_DRAWING_CNT_RATIO',
'ATM_DRAWING_CNT_RATIO', 'ISNULL_DRAWINGS_CURRENT_FLAG',
'AMT_BALANCE_YOY_PCT_CHANGE',
'AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE',
'AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE',
'AMT_DRAWINGS_ATM_CURRENT_YOY_PCT_CHANGE',
'AMT_DRAWINGS_POS_CURRENT_YOY_PCT_CHANGE',
'AMT_DRAWINGS_OTHER_CURRENT_YOY_PCT_CHANGE',
'AMT_PAYMENT_CURRENT_YOY_PCT_CHANGE',
'AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE',
'AMT_RECEIVABLE_PRINCIPAL_YOY_PCT_CHANGE',
'AMT_RECIVABLE_YOY_PCT_CHANGE',
'AMT_TOTAL_RECEIVABLE_YOY_PCT_CHANGE',
'UTILIZATION_RATIO_YOY_PCT_CHANGE', 
'PAYMENT_RATIO_YOY_PCT_CHANGE',
'DRAWING_TO_PYMNT_RATIO_YOY_PCT_CHANGE',
'INT_COVERAGE_RATIO_YOY_PCT_CHANGE',
'POS_DRAWING_AMT_RATIO_YOY_PCT_CHANGE',
'ATM_DRAWING_AMT_RATIO_YOY_PCT_CHANGE',
'POS_DRAWING_CNT_RATIO_YOY_PCT_CHANGE',
'ATM_DRAWING_CNT_RATIO_YOY_PCT_CHANGE',
'CNT_DRAWINGS_ATM_CURRENT_YOY_ABS_CHANGE',
'CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE',
'CNT_DRAWINGS_OTHER_CURRENT_YOY_ABS_CHANGE',
'CNT_DRAWINGS_POS_CURRENT_YOY_ABS_CHANGE',
'CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE', 
'SK_DPD', 'SK_DPD_DEF'
]

Last 3 Year aggregates

In [92]:
def aggregate_features(df, group_col, agg_cols, agg_funcs=('mean', 'max', 'min')):
    """
    Aggregates selected columns using specified functions.

    Args:
        df (pd.DataFrame): Input dataframe
        group_col (str): Column to group by (e.g., 'SK_ID_CURR')
        cols (list): List of columns to aggregate
        agg_funcs (tuple/list): Aggregations to apply

    Returns:
        pd.DataFrame: Aggregated dataframe with renamed columns
    """

    # Work on a copy to avoid modifying original data
    df = df[[group_col] + agg_cols].copy()

    # Perform aggregation
    agg_df = df.groupby(group_col)[agg_cols].agg(agg_funcs)

    # Reset index so it's merge-friendly
    agg_df = agg_df.reset_index()

    return agg_df

In [ ]:
CCBal_3Y_Avg = aggregate_features(CCBal_Mon[CCBal_Mon['MONTHS_BALANCE']>=-36], 'SK_ID_CURR', agg_cols, agg_funcs='mean')
CCBal_3Y_Avg.columns = [f"{col}_3Y_AVG" if col != 'SK_ID_CURR'  else col for col in CCBal_3Y_Avg.columns]

In [115]:
CCBal_3Y_Max = aggregate_features(CCBal_Mon[CCBal_Mon['MONTHS_BALANCE']>=-36], 'SK_ID_CURR', agg_cols, agg_funcs='max')
CCBal_3Y_Max.columns = [f"{col}_3Y_MAX" if col != 'SK_ID_CURR'  else col for col in CCBal_3Y_Max.columns]

1 Year Aggregates

In [116]:
CCBal_1Y_Avg = aggregate_features(CCBal_Mon[CCBal_Mon['MONTHS_BALANCE']>=-12], 'SK_ID_CURR', agg_cols, agg_funcs='mean')
CCBal_1Y_Avg.columns = [f"{col}_1Y_AVG" if col != 'SK_ID_CURR'  else col for col in CCBal_1Y_Avg.columns]

In [117]:
CCBal_1Y_Max = aggregate_features(CCBal_Mon[CCBal_Mon['MONTHS_BALANCE']>=-12], 'SK_ID_CURR', agg_cols, agg_funcs='max')
CCBal_1Y_Max.columns = [f"{col}_1Y_MAX" if col != 'SK_ID_CURR'  else col for col in CCBal_1Y_Max.columns]

Quarter Average

In [112]:
CCBal_3M_Avg = aggregate_features(CCBal_Mon[CCBal_Mon['MONTHS_BALANCE']>=-3], 'SK_ID_CURR', agg_cols, agg_funcs='mean')
CCBal_3M_Avg.columns = [f"{col}_3M_AVG" if col != 'SK_ID_CURR'  else col for col in CCBal_3M_Avg.columns]

In [170]:
CCBal_3M_Max = aggregate_features(CCBal_Mon[CCBal_Mon['MONTHS_BALANCE']>=-3], 'SK_ID_CURR', agg_cols, agg_funcs='max')
CCBal_3M_Max.columns = [f"{col}_3M_MAX" if col != 'SK_ID_CURR'  else col for col in CCBal_3M_Max.columns]

Last Snapshot

In [ ]:
CCBal_Last_Snapshot= CCBal_Mon.groupby(['SK_ID_CURR'])[agg_cols+'MONTHS_BALANCE'].last().reset_index()
CCBal_Last_Snapshot.columns = [col + 'LAST_MONTH' if col != 'SK_ID_CURR' else col for col in CCBal_Last_Snapshot.columns]
CCBal_Last_Snapshot.rename(columns={'MONTHS_BALANCE':'DAYS_SINCE_LAST_TRANSACTION'}, inplace=True)

First Snapshot

In [ ]:
#CCBal_First_Snapshot= CCBal.groupby(['SK_ID_PREV', 'SK_ID_CURR']).first().reset_index()[agg_cols]
#CCBal_First_Snapshot.columns = ['CREDITCARD_FIRST_MONTH_' + col if col not in ['SK_ID_CURR', 'SK_ID_PREV'] else col for col in CCBal_First_Snapshot.columns]

Create days to DPD

In [125]:
# create days to DPD > 1
CCBal_DPDge1 = CCBal_Mon[CCBal_Mon['SK_DPD'] >= 1].groupby(['SK_ID_CURR']).first().reset_index()[['SK_ID_CURR','MONTHS_BALANCE']] .rename(columns={'MONTHS_BALANCE': 'MONTH_DPD_GE1'}) 

In [ ]:
CCBal_First_Snapshot= CCBal_Mon.groupby(['SK_ID_CURR'])[['MONTHS_BALANCE']].first().reset_index()
CCBal_DPDge1 = CCBal_DPDge1.merge(CCBal_First_Snapshot, 
                                  on=['SK_ID_CURR'], how='left')
CCBal_DPDge1['DAYS_TO_DPD_GE1'] = CCBal_DPDge1['MONTH_DPD_GE1'] - CCBal_DPDge1['MONTHS_BALANCE']

In [ ]:
CCBal_installment_feature = (
    CCBal_Mon.groupby('SK_ID_CURR')['CNT_INSTALMENT_MATURE_CUM']
      .apply(lambda x: (x.diff() > 0).mean())
      .reset_index(name='INSTALLMENT_EVENT_RATE_AVG')
) 

Merge All data

In [179]:
CCBal_Aggregate = CCBal_3Y_Avg.merge(CCBal_3Y_Max, on=['SK_ID_CURR'], how='left')

In [180]:
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_1Y_Avg, on=['SK_ID_CURR'], how='left')
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_1Y_Max, on=['SK_ID_CURR'], how='left')

In [181]:
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_3M_Avg, on=['SK_ID_CURR'], how='left')
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_3M_Max, on=['SK_ID_CURR'], how='left')

In [182]:
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_Last_Snapshot, on=['SK_ID_CURR'], how='left')

In [ ]:
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_DPDge1[['SK_ID_CURR','DAYS_TO_DPD_GE1']], on=['SK_ID_CURR'], how='left')
CCBal_Aggregate['DAYS_TO_DPD_GE1'] = CCBal_Aggregate['DAYS_TO_DPD_GE1'].replace(np.nan, 10000)
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_installment_feature, on=['SK_ID_CURR'], how='left')

In [ ]:
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_Prev_ID_Counts, on=['SK_ID_CURR'], how='left') 

In [185]:
CCBal_Aggregate.columns.values

array(['SK_ID_CURR', 'AMT_BALANCE_3Y_AVG',
       'AMT_CREDIT_LIMIT_ACTUAL_3Y_AVG',
       'AMT_DRAWINGS_ATM_CURRENT_3Y_AVG', 'AMT_DRAWINGS_CURRENT_3Y_AVG',
       'AMT_DRAWINGS_OTHER_CURRENT_3Y_AVG',
       'AMT_DRAWINGS_POS_CURRENT_3Y_AVG',
       'AMT_INST_MIN_REGULARITY_3Y_AVG', 'AMT_PAYMENT_CURRENT_3Y_AVG',
       'AMT_PAYMENT_TOTAL_CURRENT_3Y_AVG',
       'AMT_RECEIVABLE_PRINCIPAL_3Y_AVG', 'AMT_RECIVABLE_3Y_AVG',
       'AMT_TOTAL_RECEIVABLE_3Y_AVG', 'CNT_DRAWINGS_ATM_CURRENT_3Y_AVG',
       'CNT_DRAWINGS_CURRENT_3Y_AVG', 'CNT_DRAWINGS_OTHER_CURRENT_3Y_AVG',
       'CNT_DRAWINGS_POS_CURRENT_3Y_AVG',
       'CNT_INSTALMENT_MATURE_CUM_3Y_AVG', 'UTILIZATION_RATIO_3Y_AVG',
       'PAYMENT_RATIO_3Y_AVG', 'DRAWING_TO_PYMNT_RATIO_3Y_AVG',
       'INT_COVERAGE_RATIO_3Y_AVG', 'POS_DRAWING_AMT_RATIO_3Y_AVG',
       'ATM_DRAWING_AMT_RATIO_3Y_AVG', 'POS_DRAWING_CNT_RATIO_3Y_AVG',
       'ATM_DRAWING_CNT_RATIO_3Y_AVG',
       'ISNULL_DRAWINGS_CURRENT_FLAG_3Y_AVG',
       'AMT_BALANCE_YOY_PCT_

Join Default Indicator

In [186]:
CCBal_Aggregate = CCBal_Aggregate.merge(base, on='SK_ID_CURR', how='left')

In [190]:
CCBal_Aggregate['DAYS_TO_DPD_GE1'] = CCBal_Aggregate['DAYS_TO_DPD_GE1'].replace(np.nan, 10000)

In [ ]:
CCBal_Aggregate.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\creditcard_processed.parquet")